# 04 – CDC Generate

Generates simulated CDC batches from bronze, writes them to the landing volume as JSON lines, then checks the landing files.

## Import

Imports `src/cdc_generator.py` into the notebook.

Expected output: `cwd` is the notebook folder (`.../notebooks`), `src path` is `src` at the repo root, and `LANDING_PATH` is printed.

In [0]:
import os
import sys
import importlib

notebook_dir = os.getcwd()
repo_root = os.path.dirname(notebook_dir)
src_path = os.path.join(repo_root, "src")

if src_path not in sys.path:
    sys.path.append(src_path)

import cdc_generator
importlib.reload(cdc_generator)

print("cwd:", notebook_dir)
print("src path:", src_path)
print("LANDING_PATH:", cdc_generator.LANDING_PATH)

## Parameters

`NUM_BATCHES`: number of batches to generate. `SEED`: random seed; the same value produces the same files.

In [0]:
NUM_BATCHES = 3
SEED = 42

## Generate batches and write to landing

Generates `NUM_BATCHES` batches with `run` and writes them as JSON lines under `landing/<table>/batch_<NNNN>/`. Re-running overwrites the same files.

Order in each batch: generate changes → add edge cases → apply to the in-memory state (the next batch is generated from the updated state).

Expected output: 12 write lines (3 batches × 4 tables). Edge case IDs: 435, 4206, 1253, 101254, 107308. At the end, each table folder lists 3 batch folders.

In [0]:
cdc_generator.run(spark, NUM_BATCHES, SEED)

print()
for table_name in ["client", "account", "card", "loan"]:
    table_path = f"{cdc_generator.LANDING_PATH}/{table_name}/"
    batch_folders = []
    for file_info in dbutils.fs.ls(table_path):
        batch_folders.append(file_info.name)
    print(table_name, batch_folders)

## Landing checks

**1) Counts by table / batch / op_type**

The JSON files of the 4 tables are read with Spark and kept in the `landing_dfs` dictionary; the next check cells reuse it.

Each output row is one table + batch. The `I`, `U`, `D` columns show op counts; an empty cell means that op does not occur in that batch. `total` should match the row counts printed by the generator.

In [0]:
from pyspark.sql import functions as F

CDC_TABLES = ["client", "account", "card", "loan"]

landing_dfs = {}
df_counts = None

for table_name in CDC_TABLES:
    path = f"{cdc_generator.LANDING_PATH}/{table_name}/batch_*/"
    df_table = spark.read.json(path)
    landing_dfs[table_name] = df_table

    df_table_counts = df_table.groupBy("batch_id", "op_type").count()
    df_table_counts = df_table_counts.withColumn("table_name", F.lit(table_name))

    if df_counts is None:
        df_counts = df_table_counts
    else:
        df_counts = df_counts.unionByName(df_table_counts)

df_pivot = df_counts.groupBy("table_name", "batch_id").pivot("op_type", ["I", "U", "D"]).sum("count")
df_pivot = df_pivot.withColumn("total", F.coalesce(F.col("I"), F.lit(0)) + F.coalesce(F.col("U"), F.lit(0)) + F.coalesce(F.col("D"), F.lit(0)))

display(df_pivot.orderBy("table_name", "batch_id"))

**2) `change_ts` ranges**

Earliest and latest `change_ts` per table and batch. `change_ts` is an ISO string in the files and is cast to timestamp here.

Expected: batch 1 → 1999-01, batch 2 → 1999-02, batch 3 → 1999-03. `in_batch_month` is `true` when both min and max fall in the batch month. `null_ts` counts values that could not be cast to timestamp; it should be 0.

In [0]:
df_ts_ranges = None

for table_name in CDC_TABLES:
    df_table = landing_dfs[table_name]
    df_table = df_table.withColumn("change_ts_ts", F.to_timestamp("change_ts"))

    df_table_ranges = df_table.groupBy("batch_id").agg(
        F.min("change_ts_ts").alias("min_ts"),
        F.max("change_ts_ts").alias("max_ts"),
        F.sum(F.col("change_ts_ts").isNull().cast("int")).alias("null_ts"),
    )
    df_table_ranges = df_table_ranges.withColumn("table_name", F.lit(table_name))

    if df_ts_ranges is None:
        df_ts_ranges = df_table_ranges
    else:
        df_ts_ranges = df_ts_ranges.unionByName(df_table_ranges)

min_ok = (F.year("min_ts") == 1999) & (F.month("min_ts") == F.col("batch_id"))
max_ok = (F.year("max_ts") == 1999) & (F.month("max_ts") == F.col("batch_id"))
df_ts_ranges = df_ts_ranges.withColumn("in_batch_month", min_ok & max_ok)

display(df_ts_ranges.select("table_name", "batch_id", "min_ts", "max_ts", "null_ts", "in_batch_month").orderBy("table_name", "batch_id"))

**3) Batch 1 consistency with bronze**

Batch 1 was generated directly from bronze, so each record is joined to bronze on the key, and the old (bronze) and new (landing) values of the changed column are compared.

Columns:

- `ud_key_missing`: U/D for a key not in bronze → **0**
- `i_key_exists`: I for a key already in bronze → **0**
- `u_not_changed`: U with an unchanged value → **0**
- `d_differs`: D that differs from the last bronze version → **0**
- `u_old_values` / `u_new_values`: old and new values of U records (card: `classic` → `gold`, loan: `C` → `A`/`D`)

In [0]:
TRACKED_COLUMNS = {
    "client": "district_id",
    "account": "frequency",
    "card": "type",
    "loan": "status",
}

df_bronze_checks = None

for table_name in CDC_TABLES:
    key_col = cdc_generator.TABLE_KEYS[table_name]
    value_col = TRACKED_COLUMNS[table_name]

    df_bronze = spark.table(f"{cdc_generator.BRONZE_SCHEMA}.{table_name}")
    df_bronze = df_bronze.select(key_col, F.col(value_col).alias("old_value"))

    df_batch1 = landing_dfs[table_name].filter(F.col("batch_id") == 1)
    df_batch1 = df_batch1.select(key_col, "op_type", F.col(value_col).alias("new_value"))

    df_joined = df_batch1.join(df_bronze, on=key_col, how="left")

    is_u = F.col("op_type") == "U"
    is_d = F.col("op_type") == "D"
    is_i = F.col("op_type") == "I"
    key_missing = F.col("old_value").isNull()

    df_check = df_joined.agg(
        F.count("*").alias("rows"),
        F.count_if((is_u | is_d) & key_missing).alias("ud_key_missing"),
        F.count_if(is_i & ~key_missing).alias("i_key_exists"),
        F.count_if(is_u & (F.col("new_value") == F.col("old_value"))).alias("u_not_changed"),
        F.count_if(is_d & (F.col("new_value") != F.col("old_value"))).alias("d_differs"),
        F.array_sort(F.collect_set(F.when(is_u, F.col("old_value")))).alias("u_old_values"),
        F.array_sort(F.collect_set(F.when(is_u, F.col("new_value")))).alias("u_new_values"),
    )
    df_check = df_check.withColumn("table_name", F.lit(table_name))

    if df_bronze_checks is None:
        df_bronze_checks = df_check
    else:
        df_bronze_checks = df_bronze_checks.unionByName(df_check)

display(df_bronze_checks.select("table_name", "rows", "ud_key_missing", "i_key_exists", "u_not_changed", "d_differs", "u_old_values", "u_new_values"))

**4) Edge case records (seed 42)**

The edge case IDs printed by the generator are looked up in landing. `in_bronze` shows whether the key exists in bronze.

Expected:

- client 435 and 4206: two U in batch 2, the second with a later `change_ts` (U → U)
- card 1253: in batch 2, first I `classic`, then U `gold` (I → U); not in bronze (new card)
- card 101254 (D) and loan 107308 (U): in batch 3; not in bronze (missing key)

In [0]:
EDGE_CASE_IDS = {
    "client": ["435", "4206"],
    "card": ["1253", "101254"],
    "loan": ["107308"],
}

df_edge_cases = None

for table_name in EDGE_CASE_IDS:
    key_col = cdc_generator.TABLE_KEYS[table_name]
    value_col = TRACKED_COLUMNS[table_name]
    edge_ids = EDGE_CASE_IDS[table_name]

    df_bronze_keys = spark.table(f"{cdc_generator.BRONZE_SCHEMA}.{table_name}").select(key_col)
    df_bronze_keys = df_bronze_keys.withColumn("in_bronze", F.lit(True))

    df_edge = landing_dfs[table_name].filter(F.col(key_col).isin(edge_ids))
    df_edge = df_edge.join(df_bronze_keys, on=key_col, how="left")
    df_edge = df_edge.select(
        F.lit(table_name).alias("table_name"),
        F.col(key_col).alias("id"),
        "batch_id",
        "op_type",
        "change_ts",
        F.col(value_col).alias("value"),
        F.coalesce(F.col("in_bronze"), F.lit(False)).alias("in_bronze"),
    )

    if df_edge_cases is None:
        df_edge_cases = df_edge
    else:
        df_edge_cases = df_edge_cases.unionByName(df_edge)

display(df_edge_cases.orderBy("table_name", "id", "batch_id", "change_ts"))